# Healthcare Liver Disease Decision Support — Milestone 2
**Participant:** Participant_M2_01  
**Project:** Healthcare Liver Disease Decision Support  
**Milestone:** 2  
**Dataset package:** `HealthCareData.zip`

## Objective
Predict liver disease status for educational decision support, explain model factors, and communicate risk responsibly.

## Milestone scope
Decision Tree, Random Forest, and boosting classification; tuning and cross-validation; compare metrics; feature importance; illustrative 0–100 risk score with non-clinical risk bands.

> This is a synthetic sample submission for evaluator testing. Dataset files must be placed in the paths referenced below. Results are not pre-filled or fabricated.

## ML classifiers and tuning
Use stratified cross-validation on the training set. Do not use test data to choose hyperparameters. False negatives are important to inspect for this educational screening task.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

df = pd.read_csv(Path("data") / "liver_patient_joined.csv")
y = df["Status"].map({1: 1, 2: 0})
X = df.drop(columns=["Status", "ID"], errors="ignore")
cat = X.select_dtypes(include=["object", "category"]).columns.tolist()
num = X.select_dtypes(include=np.number).columns.tolist()
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat)
])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
candidates = {
    "DecisionTree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42)
}
rows = []
for name, est in candidates.items():
    pipe = Pipeline([("prep", prep), ("model", est)])
    pipe.fit(Xtr, ytr)
    prob = pipe.predict_proba(Xte)[:, 1] if hasattr(pipe, "predict_proba") else None
    pred = pipe.predict(Xte)
    rows.append({"model": name, "ROC_AUC": roc_auc_score(yte, prob) if prob is not None else np.nan,
                 "recall": __import__("sklearn.metrics").metrics.recall_score(yte, pred),
                 "f1": __import__("sklearn.metrics").metrics.f1_score(yte, pred)})
display(pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False))

## Illustrative risk score
A predicted probability multiplied by 100 is an illustrative score, not a clinically validated threshold. Document calibration and avoid describing risk bands as diagnosis.

In [ ]:
# After selecting a model on training/CV results, fit it and inspect test probabilities.
# Example mapping only; the bands are demonstrative and require domain review.
def illustrative_band(score):
    if score < 30: return "lower illustrative score"
    if score < 70: return "middle illustrative score"
    return "higher illustrative score"